# 9.1 Agenda Scraping — Trump & Harris Policy Platforms

Scrapes and structures the official policy platforms of both candidates.

**Sources:**
- **Harris** — `kamalaharris.com/issues/` via Wayback Machine (archived 3 Nov 2024, day before election)
- **Trump** — `en.wikipedia.org/wiki/Political_positions_of_Donald_Trump` (structured policy sections)
- **Trump (campaign)** — `ballotpedia.org/Donald_Trump_presidential_campaign,_2024`

**Output:** `Data/agenda_keywords.json` — structured topic → keywords mapping used by downstream analysis notebooks.


<!-- toc -->
## Contents
- **[1. Setup](#1-setup)**
- **[2. Scrape Harris Platform](#2-scrape-harris-platform)**
- **[3. Scrape Trump Platform](#3-scrape-trump-platform)**
- **[4. Build Keyword Dictionary](#4-build-keyword-dictionary)**
- **[5. Save & Inspect](#5-save--inspect)**


## 1. Setup


In [2]:
import sys, re, json, time
sys.path.insert(0, '../..')

import requests
from bs4 import BeautifulSoup
from pathlib import Path
from collections import defaultdict

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

OUT_PATH = Path('../../Data/agenda_keywords.json')

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

def fetch(url, timeout=15):
    """GET with retries. Returns BeautifulSoup or raises."""
    for attempt in range(3):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            r.raise_for_status()
            return BeautifulSoup(r.text, 'html.parser')
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(2)
    raise RuntimeError(f'Could not fetch {url}')

print('Setup done.')


Setup done.


## 2. Scrape Harris Platform


In [3]:
# Archived campaign issues page (day before election, 3 Nov 2024)
HARRIS_URL = 'https://web.archive.org/web/20241103120000/https://kamalaharris.com/issues/'

print(f'Fetching: {HARRIS_URL}')
soup_h = fetch(HARRIS_URL)
print('OK —', soup_h.title.text.strip() if soup_h.title else 'no title')


Fetching: https://web.archive.org/web/20241103120000/https://kamalaharris.com/issues/
OK — Issues - Kamala Harris for President: Official Campaign Website


In [4]:
harris_sections = {}

# Map the 4 top-level h2 sections to cleaner topic names
HARRIS_TOPIC_MAP = {
    'BUILD AN OPPORTUNITY ECONOMY':  'economy',
    'SAFEGUARD OUR FUNDAMENTAL':     'freedoms',
    'ENSURE SAFETY AND JUSTICE':     'safety_justice',
    'KEEP AMERICA SAFE, SECURE':     'foreign_security',
}

for h2 in soup_h.find_all('h2'):
    title_raw = h2.get_text(strip=True)
    if len(title_raw) < 5 or 'UPDATED' in title_raw:
        continue

    # Match to clean topic name
    topic = next((v for k, v in HARRIS_TOPIC_MAP.items() if title_raw.startswith(k)), None)
    if topic is None:
        topic = re.sub(r'[^a-z0-9]+', '_', title_raw.lower())[:40]

    # Collect all text in this section until next h2
    paragraphs = []
    for sib in h2.find_next_siblings():
        if sib.name == 'h2':
            break
        t = sib.get_text(' ', strip=True)
        if len(t) > 30:
            paragraphs.append(t)

    if paragraphs:
        harris_sections[topic] = {
            'title':  title_raw.title(),
            'text':   ' '.join(paragraphs),
        }
        print(f'  [{topic}]  {len(" ".join(paragraphs).split())} words')

print(f'\nHarris: {len(harris_sections)} sections scraped.')


  [economy]  2512 words
  [freedoms]  605 words
  [safety_justice]  990 words
  [foreign_security]  902 words

Harris: 4 sections scraped.


## 3. Scrape Trump Platform


In [5]:
# Wikipedia: Political positions of Donald Trump
TRUMP_WIKI = 'https://en.wikipedia.org/wiki/Political_positions_of_Donald_Trump'

print(f'Fetching: {TRUMP_WIKI}')
soup_t = fetch(TRUMP_WIKI)
print('OK —', soup_t.title.text.strip() if soup_t.title else 'no title')


Fetching: https://en.wikipedia.org/wiki/Political_positions_of_Donald_Trump
OK — Political positions of Donald Trump - Wikipedia


In [6]:
# Skip boilerplate sections
SKIP = {
    'Contents', 'References', 'Notes', 'See also', 'External links',
    'Further reading', 'Navigation menu', 'Political affiliation and ideology',
    'Self-described', 'As described by others', 'Scales and rankings',
    'Politics and policies during presidency',
}

# Map broad section headings to clean topic keys
TRUMP_TOPIC_MAP = {
    'Domestic policy':              'domestic',
    'Economy and trade':            'economy_trade',
    'Environment and energy':       'energy_environment',
    'Foreign policy and defense':   'foreign_defense',
    'Immigration':                  'immigration',
    'Healthcare':                   'healthcare',
    'Education':                    'education',
    'Civil rights':                 'civil_rights',
    'Crime and law enforcement':    'crime',
    'Gun policy':                   'guns',
    'Abortion':                     'abortion',
    'Social Security':              'social_security',
    'Taxes':                        'taxes',
    'Trade':                        'trade',
    'Climate change':               'climate',
}

trump_sections = {}
content_div = soup_t.find('div', {'id': 'mw-content-text'})

current_topic = None
current_texts = []

def flush(topic, texts):
    if topic and texts:
        trump_sections[topic] = {
            'title': topic.replace('_', ' ').title(),
            'text':  ' '.join(texts),
        }

for tag in content_div.find_all(['h2', 'h3', 'p']):
    if tag.name in ('h2', 'h3'):
        raw = tag.get_text(strip=True).replace('[edit]', '').strip()
        if raw in SKIP:
            flush(current_topic, current_texts)
            current_topic = None
            current_texts = []
            continue
        # Match to clean topic key
        matched = next((v for k, v in TRUMP_TOPIC_MAP.items()
                        if raw.lower().startswith(k.lower())), None)
        new_topic = matched if matched else re.sub(r'[^a-z0-9]+', '_', raw.lower())[:40]
        if new_topic != current_topic:
            flush(current_topic, current_texts)
            current_topic = new_topic
            current_texts = []
    elif tag.name == 'p' and current_topic:
        t = tag.get_text(' ', strip=True)
        if len(t) > 40:
            current_texts.append(t)

flush(current_topic, current_texts)

# Remove very short sections
trump_sections = {
    k: v for k, v in trump_sections.items()
    if len(v['text'].split()) >= 50
}

for k, v in trump_sections.items():
    print(f'  [{k}]  {len(v["text"].split())} words')
print(f'\nTrump: {len(trump_sections)} sections scraped.')


  [campaign_finance]  212 words
  [civil_servants]  68 words
  [disabled_people]  54 words
  [education]  440 words
  [food_safety]  83 words
  [native_americans]  398 words
  [questioning_obama_s_citizenship]  526 words
  [social_security]  246 words
  [veterans]  853 words
  [economy_trade]  55 words
  [trade]  133 words
  [cryptocurrencies]  112 words
  [energy_environment]  66 words
  [california_drought]  109 words
  [climate]  1343 words
  [energy_independence]  72 words
  [environmental_regulation]  210 words
  [pipelines]  261 words
  [renewable_energy]  357 words
  [wildlife_conservation_and_animal_welfare]  142 words
  [health_care]  63 words
  [2016_campaign]  129 words
  [actions_while_in_office_2017_2021_]  1301 words
  [2024_campaign]  133 words
  [public_health]  483 words
  [immigration]  183 words
  [capital_punishment]  285 words
  [criminal_justice]  568 words
  [drug_policy]  385 words
  [gun_regulation]  1069 words
  [judiciary]  627 words
  [term_limits_and_ethics

## 4. Build Keyword Dictionary


For each policy section, extract the **top-20 TF-IDF keywords** across all sections of that candidate.
This gives keywords that are distinctive for each topic (not just frequent words that appear everywhere).


In [7]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Extra stopwords specific to political boilerplate
EXTRA_STOPS = [
    'harris', 'trump', 'president', 'vice', 'kamala', 'donald',
    'will', 'has', 'have', 'american', 'america', 'united', 'states',
    'also', 'said', 'would', 'administration', 'presidential',
    'campaign', 'election', 'candidate', 'policy', 'policies',
]

def extract_keywords(sections, top_n=20):
    """TF-IDF across sections of one candidate; return top_n keywords per section."""
    keys   = list(sections.keys())
    corpus = [sections[k]['text'] for k in keys]

    vec = TfidfVectorizer(
        max_features=5000,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
    )
    X = vec.fit_transform(corpus)
    feature_names = vec.get_feature_names_out()

    result = {}
    for i, k in enumerate(keys):
        row    = X[i].toarray().flatten()
        top_idx = row.argsort()[::-1][:top_n * 2]  # get extra, then filter
        keywords = []
        for idx in top_idx:
            word = feature_names[idx]
            # Filter: only single words or clean bigrams, no boilerplate
            parts = word.split()
            if any(p in EXTRA_STOPS for p in parts):
                continue
            if any(len(p) < 3 for p in parts):
                continue
            keywords.append(word)
            if len(keywords) == top_n:
                break
        result[k] = keywords
    return result

harris_kw = extract_keywords(harris_sections)
trump_kw  = extract_keywords(trump_sections)

print('=== Harris keywords per section ===')
for topic, kws in harris_kw.items():
    print(f'  {topic:25s}: {", ".join(kws[:10])}')

print('\n=== Trump keywords per section ===')
for topic, kws in trump_kw.items():
    print(f'  {topic:25s}: {", ".join(kws[:10])}')


=== Harris keywords per section ===
  economy                  : class, middle class, tax, small, million, jobs, affordable, costs, big, prices
  freedoms                 : abortion, reproductive, voting, love, women, lgbtqi, roe, abortion bans, discrimination, reproductive freedom
  safety_justice           : border, gun, violence, fentanyl, gun violence, immigration, communities, safe, violent, crime
  foreign_security         : stand, leaders, national security, veterans, russia, israel, military, members veterans, stage, world stage

=== Trump keywords per section ===
  campaign_finance         : 107, pacs, super pacs, super, race, early 2016, 113, finance, early, law
  civil_servants           : deconstruct administrative, bannon stated, briefly, briefly leader, chief strategist, christie, christie served, according chris, easier public, administrative
  disabled_people          : disability, 119, disability related, disability group, disabled people, website mention, responded, 1

## 5. Save & Inspect


In [8]:
agenda = {
    'harris': {
        topic: {
            'title':    harris_sections[topic]['title'],
            'keywords': harris_kw[topic],
            'text':     harris_sections[topic]['text'],
        }
        for topic in harris_sections
    },
    'trump': {
        topic: {
            'title':    trump_sections[topic]['title'],
            'keywords': trump_kw[topic],
            'text':     trump_sections[topic]['text'],
        }
        for topic in trump_sections
    },
}

with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(agenda, f, ensure_ascii=False, indent=2)

print(f'Saved to {OUT_PATH}')
print(f'  Harris sections : {len(agenda["harris"])}')
print(f'  Trump  sections : {len(agenda["trump"])}')
print(f'  File size       : {OUT_PATH.stat().st_size / 1024:.1f} KB')


Saved to ..\..\Data\agenda_keywords.json
  Harris sections : 4
  Trump  sections : 50
  File size       : 155.1 KB


In [9]:
# Summary table
rows = []
for cand in ('harris', 'trump'):
    for topic, data in agenda[cand].items():
        rows.append({
            'candidate': cand.title(),
            'topic':     topic,
            'title':     data['title'],
            'n_keywords': len(data['keywords']),
            'top_5_keywords': ', '.join(data['keywords'][:5]),
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))


candidate                                    topic                                                     title  n_keywords                                                                       top_5_keywords
   Harris                                  economy Build An Opportunity Economy And Lower Costs For Families          20                                             class, middle class, tax, small, million
   Harris                                 freedoms                        Safeguard Our Fundamental Freedoms          20                                          abortion, reproductive, voting, love, women
   Harris                           safety_justice                         Ensure Safety And Justice For All          20                                        border, gun, violence, fentanyl, gun violence
   Harris                         foreign_security                 Keep America Safe, Secure, And Prosperous          20                                  stand, leaders, nation